# Bitcoin Heist : détection d'adresses associées aux rançongiciels
**Auteurs du projet :** Théo MICHEL & Théo RICO — ISAE-SUPAERO

---

## 1. Contexte, problématique et hypothèses

Un rançongiciel est un logiciel malveillant qui bloque l'accès à des données ou à des ressources et réclame une rançon. Bitcoin facilite les paiements à distance, mais ses transactions sont publiques et les adresses sont pseudonymes : on peut analyser leurs flux sans connaître directement l'identité des personnes qui les contrôlent.

L'article *BitcoinHeist: Topological Data Analysis for Ransomware Detection on the Bitcoin Blockchain* étudie des caractéristiques extraites du graphe Bitcoin et compare notamment des méthodes de similarité, DBSCAN, Random Forest, XGBoost et Mapper (TDA). Il distingue deux tâches : retrouver de nouvelles adresses d'une famille déjà connue et signaler des adresses d'une famille encore inconnue. Les résultats varient selon les familles et comportent des faux positifs; `white` doit donc être interprété comme « non étiqueté ransomware dans les sources », et non comme une preuve d'innocence.

**Périmètre du fichier fourni.** L'article rapporte un jeu combiné de 24 486 adresses réparties entre 27 familles; le CSV de ce projet contient 41 413 observations étiquetées ransomware, 20 849 adresses positives distinctes et 28 familles positives. Les unités d'observation et les effectifs diffèrent : l'analyse ci-dessous porte sur le CSV fourni et ne prétend pas reproduire exactement les résultats de l'article.

Le fichier fourni contient des agrégats quotidiens par adresse, et non les transactions brutes ni le graphe complet. Ce notebook conserve une expérience initiale sur des adresses non vues, puis organise l'amélioration des modèles et une extension à une période future. Il ne prétend ni anticiper un paiement avant l'observation des caractéristiques de sa journée, ni reproduire exactement le protocole TDA de l'article.

**Problématique générale :** les caractéristiques topologiques quotidiennes permettent-elles de repérer des observations associées à des familles de rançongiciels déjà connues, lorsque l'apprentissage précède la période évaluée?

**Hypothèses exploratoires :** les distributions ou combinaisons de `length`, `weight`, `count`, `looped`, `neighbors` et `income` peuvent différer entre observations `white` et observations étiquetées ransomware. Ces écarts doivent être vérifiés sur des périodes distinctes et ne démontrent pas, à eux seuls, qu'une adresse est malveillante.


### 1.1 Objectif opérationnel et niveau de preuve

**Première problématique : les six caractéristiques d'activité permettent-elles de retrouver des observations étiquetées ransomware sur des adresses absentes de l'entraînement ?** La généralisation dans le temps et la découverte de familles inconnues seront évaluées dans des expériences distinctes.

L'usage envisagé est la **priorisation d'investigations**. Une alerte signale une observation à examiner ; elle ne constitue pas une preuve de malveillance. Avant de retenir un seuil de décision, il faudra convenir d'une précision minimale ou d'une capacité quotidienne d'investigation. Un budget de 50 adresses par jour, par exemple, serait une hypothèse de travail à valider et non une exigence déjà établie.

Nous faisons trois hypothèses : les distributions des caractéristiques peuvent différer entre classes ; leurs combinaisons peuvent porter un signal non linéaire ; une partie de ce signal peut se retrouver sur des adresses non vues. Ces hypothèses sont examinées à partir des résultats, sans présumer de la performance.

Cette étude relève du **Machine Learning (apprentissage automatique)**, une composante de l'intelligence artificielle (AI). L'apprentissage est **supervisé** parce que les labels interviennent dans l'entraînement. Le Deep Learning (DL) désigne des modèles à réseaux de neurones profonds ; les données tabulaires et le protocole retenu ne nécessitent pas ici cette complexité supplémentaire. Une analyse **non supervisée** cherchera plutôt des structures ou des observations atypiques sans utiliser les labels pour apprendre.


### 1.2 Organisation et plan d'étude

| Partie | Question traitée | 
|---|---|
| 2. Données et intégrité | Que contient le fichier et que peut-on en déduire ? | 
| 3. Exploration | Quels déséquilibres et associations orientent l'analyse ? | 
| 4. Construction des variables | Comment représenter l'activité sans introduire d'information future ? | 
| 5. Apprentissage | Que vaut la première forêt sur des adresses non vues ? |
| 6–7. Amélioration et validation | Quels effets viennent du seuil et des hyperparamètres ? | 
| 8. Comparaison XGBoost | Un autre ensemble d'arbres améliore-t-il le compromis retenu ? | 
| 9–11. Explication et robustesse | Quels mécanismes, périodes et familles limitent les modèles ? | 
| 12. Conclusion | Qu'avons-nous démontré et que reste-t-il à vérifier ? | 

## 2. Données, définitions et contrôles

### 2.1 Environnement et lecture du fichier


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.express as px
from IPython.display import display
from html import escape
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    precision_score, recall_score, f1_score, average_precision_score,
    precision_recall_curve,
)

seed = 45


`income` est conservé en **satoshis**, l'unité entière de la source : 100 000 000 satoshis correspondent à 1 BTC. Ce choix facilite l'audit des montants et évite de créer une colonne décimale redondante. La conversion en BTC ne détruit pas systématiquement l'information ; les arrondis d'affichage et la représentation binaire des nombres décimaux demandent simplement de la vigilance.


In [ ]:
dtypes = {
    "address": "string",
    "year": "int16",
    "day": "int16",
    "length": "int32",
    "weight": "float64",
    "count": "int32",
    "looped": "int32",
    "neighbors": "int32",
    "income": "int64",
    "label": "category",
}

data = pd.read_csv("BitcoinHeistData.csv", dtype=dtypes, low_memory=False)
print(f"Shape du dataset : {data.shape[0]:,} lignes, {data.shape[1]} colonnes")
print(f"Mémoire profonde : {data.memory_usage(index=True, deep=True).sum() / 2**20:.2f} MiB")
data.head()


### 2.2 Définition des caractéristiques

Chaque ligne décrit les agrégats d'activité d'une adresse dans une fenêtre quotidienne. Une même adresse peut donc apparaître dans plusieurs lignes à des dates différentes. Les six features ci-dessous sont déjà calculées à partir du graphe de transactions; le CSV ne fournit pas les transactions brutes nécessaires pour reconstruire ce graphe.

| Variable | Description |
|---|---|
| `address` | Identifiant public de l'adresse. À conserver pour regrouper et contrôler les réapparitions, mais à exclure des entrées du modèle. |
| `year`, `day` | Année et numéro du jour dans l'année (1 à 365 ou 366). Ensemble, ils identifient la fenêtre quotidienne. |
| `length` | Longueur du plus long chemin orienté depuis une transaction de départ (*starter transaction*) jusqu'à l'adresse, comptée en transactions non-starter. Une valeur nulle correspond à une adresse directement atteinte depuis une transaction de départ. |
| `weight` | Poids structurel propagé depuis les transactions de départ et agrégé à l'adresse. Ce n'est ni un montant ni une proportion bornée : plusieurs contributions peuvent s'additionner, donc `weight` peut dépasser 1. |
| `count` | Nombre de transactions de départ distinctes pouvant atteindre l'adresse dans le graphe de la fenêtre. Ce n'est pas le nombre total de transactions de l'adresse. |
| `looped` | Nombre de transactions de départ pouvant atteindre l'adresse par plusieurs chemins orientés. La cohérence attendue dans les données est `looped <= count`. |
| `neighbors` | Nombre de transactions qui ont l'adresse parmi leurs sorties, c'est-à-dire le nombre de transactions qui lui versent des fonds dans la fenêtre; ce n'est pas le nombre d'adresses voisines. |
| `income` | Montant total reçu dans la fenêtre, en satoshis. `100 000 000` satoshis = `1 BTC`. Ce montant n'est ni le solde ni le bénéfice de l'adresse. Le montant reste exprimé en satoshis dans cette étude. |
| `label` | Label fourni par les sources de données : `white` ou nom d'une famille de ransomware connue. C'est la cible, jamais une entrée du modèle; `white` signifie non identifié, pas sain confirmé. |

### Conséquences pour l'analyse

- Exclure `address` et `label` des features. Conserver `address` uniquement pour l'audit des observations répétées et les découpages par groupe.
- Utiliser `income` en satoshis dans les graphiques et les modèles. Une conversion ponctuelle en BTC peut servir à commenter un montant, sans devenir une seconde entrée du modèle.
- Les comptages sont non négatifs et entiers; `weight` est décimal et peut être supérieur à 1. Ne pas appliquer de règle arbitraire qui le bornerait à [0, 1].
- Les transformations logarithmiques seront comparées pour l'exploration et les modèles sensibles à l'échelle. Les arbres n'en ont généralement pas besoin; aucune transformation ne justifie à elle seule la suppression d'observations extrêmes.
- `year` et `day` servent d'abord à construire la date et à découper les données dans le temps. Les inclure comme prédicteurs pourrait permettre au modèle d'apprendre des changements historiques propres aux labels.


In [ ]:
raw_numeric_cols = [
    "year", "day", "length", "weight", "count", "looped", "neighbors", "income"
]
integer_cols = [col for col in raw_numeric_cols if col != "weight"]
required_cols = ["address", *raw_numeric_cols, "label"]
features = ["length", "weight", "count", "looped", "neighbors", "income"]

df = data.copy()
year_start = pd.to_datetime(df["year"].astype("string"), format="%Y")
df["date"] = year_start + pd.to_timedelta(df["day"] - 1, unit="D")
data_clean = df[[*required_cols, "date"]].copy()


### 2.3 Intégrité des données et audit des valeurs

Les contrôles recensent les doublons, les valeurs absentes ou non numériques, les incohérences de date et les relations attendues entre caractéristiques. Ils ne remplacent ni ne suppriment aucune observation.


In [ ]:
def _display_report(report, 
                    title, 
                    display_mode="auto"):
    """Affiche un rapport en HTML adapté pour Jupyter. Permet de rendre les 
    tableaux entièrement. """

    if display_mode not in {"auto", "html", "text", "none"}:
        raise ValueError("display_mode doit être auto, html, text ou none.")
    if display_mode == "none":
        return
    
    # Un noyau Jupyter sait afficher le HTML, contrairement à un terminal.
    use_html = display_mode == "html"
    if display_mode == "auto":
        try:
            from IPython import get_ipython
            use_html = getattr(get_ipython(), "kernel", None) is not None
        except ImportError:
            use_html = False

    # Les compteurs constituent le résumé. Les DataFrames sont les détails.
    summary = pd.DataFrame([
        {"Vérification": key, "Résultat": str(value)}
        for key, value in report.items() if not isinstance(value, pd.DataFrame)
    ])
    tables = [("Bilan", summary)] + [
        (key.replace("_", " ").capitalize(), value)
        for key, value in report.items() if isinstance(value, pd.DataFrame)
    ]
    if use_html:
        from IPython.display import HTML, display
        sections = [f"<h3>{escape(title)}</h3>"]
        for i, (name, table) in enumerate(tables):
            # Échapper les cellules évite qu'un label soit interprété comme HTML.
            # max_rows/max_cols=None garantissent un tableau sans « ... ».
            rendered = table.round(3).to_html(
                index=name != "Bilan" and "colonne" not in table.columns,
                max_rows=None, max_cols=None, escape=True, border=0,
            )
            opened = " open" if i == 0 else ""
            sections.append(
                f"<details{opened}><summary><b>{escape(name)}</b> "
                f"({len(table)} lignes)</summary>"
                '<div style="overflow:auto;max-height:420px;padding:8px;">'
                f"{rendered}</div></details>"
            )
        display(HTML("".join(sections)))
    else:
        # to_string contourne la troncature pandas et conserve l'alignement des colonnes.
        print(title)
        for name, table in tables:
            print(f"\n{name} :")
            print(table.round(3).to_string(
                index=name != "Bilan" and "colonne" not in table.columns
            ))



def check_dataset(data, 
                  reference_date="2026-10-01", 
                  display_mode="auto"):
    """Affiche et retourne un audit Bitcoin Heist, sans modifier le 
    dataset. Les conversions servent uniquement aux contrôles. Toutes les lignes,
    y compris les doublons, sont comptées. La date par défaut est celle
    fixée dans les notebooks pour la reproductibilité.

    display_mode : "auto" détecte Jupyter, "html" force les tableaux repliables,
    "text" produit un bilan texte et "none" permet un audit sans affichage.
    """
    numeric_cols = ["year", "day", "length", "weight", "count",
                    "looped", "neighbors", "income"]
    integer_cols = [col for col in numeric_cols if col != "weight"]
    required_cols = ["address", *numeric_cols, "label"]
    
    report = {}
    def record(name, value):
        report[name] = value

    record("Nombre de lignes", len(data))
    record("Nombre de colonnes", data.shape[1])
    record("Nombre de noms de colonnes dupliqués",
           int(data.columns.duplicated().sum()))
    missing_cols = [col for col in required_cols if col not in data.columns]
    record("Nombre de colonnes requises absentes", len(missing_cols))
    record("Colonnes requises absentes", missing_cols)
    record("Nombre de doublons stricts", int(data.duplicated().sum()))

    # 2. Valeurs manquantes, sur toutes les colonnes et toutes les lignes.
    missing = data.isna().copy()
    for i in range(data.shape[1]):
        blank = data.iloc[:, i].map(
            lambda value: isinstance(value, str) and not value.strip()
        )
        missing.iloc[:, i] = missing.iloc[:, i] | blank
    record("Nombre de valeurs nulles (NaN / None / NA)",
           int(data.isna().sum().sum()))
    record("Nombre de cases vides (valeurs nulles ou chaînes blanches)",
           int(missing.sum().sum()))
    record("Nombre de lignes avec au moins une case vide",
           int(missing.any(axis=1).sum()))
    audit = pd.DataFrame({
        "colonne": data.columns,
        "type": data.dtypes.astype(str).to_numpy(),
        "cases_vides": missing.sum().to_numpy(),
        "valeurs_distinctes": data.nunique().to_numpy(),
    })
    report["audit_colonnes"] = audit

    if not data.columns.is_unique:
        # Un accès data[col] serait ambigu : conserver les contrôles globaux
        # et expliquer pourquoi les contrôles suivants ne sont pas exécutés.
        record("Contrôles non exécutés", "Contrôles par nom : noms dupliqués")
        _display_report(report, "Vérification du dataset", display_mode)
        return report

    # 3. Conversion TEMPORAIRE des variables numériques pour les vérifier.
    # errors="coerce" permet de repérer les textes non convertibles. 
    numeric = pd.DataFrame(index=data.index)
    conversion_report = []
    unreadable_rows = pd.Series(False, index=data.index)
    infinite_rows = pd.Series(False, index=data.index)
    for col in numeric_cols:
        if col not in data.columns:
            continue
        converted = pd.to_numeric(data[col], errors="coerce")
        nonconvertible = ~missing[col] & converted.isna()
        infinite = converted.isin([np.inf, -np.inf])
        unreadable_rows |= nonconvertible
        infinite_rows |= infinite
        # Seule cette série temporaire exclut les infinis des calculs.
        numeric[col] = converted.where(~infinite)
        conversion_report.append({
            "colonne": col,
            "non_convertibles": int(nonconvertible.sum()),
            "infinis": int(infinite.sum()),
            "negatifs": int(converted.lt(0).sum()),
            "zeros": int(converted.eq(0).sum()),
            "entiers_fractionnaires": int(
                (numeric[col].notna() & numeric[col].mod(1).ne(0)).sum()
            ) if col in integer_cols else 0,
        })
    conversion = pd.DataFrame(conversion_report)
    report["audit_numerique"] = conversion
    record("Nombre de valeurs non convertibles",
           sum(item["non_convertibles"] for item in conversion_report))
    record("Nombre de valeurs infinies",
           sum(item["infinis"] for item in conversion_report))
    negative_rows = numeric.lt(0).any(axis=1)
    # Les infinis négatifs comptent également comme valeurs négatives.
    for col in numeric.columns:
        negative_rows |= pd.to_numeric(data[col], errors="coerce").eq(-np.inf)
    # weight accepte les décimales ; les autres variables doivent être
    # entières. Les valeurs absentes et infinies sont exclues de ce contrôle.
    integers = numeric[[col for col in integer_cols if col in numeric.columns]]
    fractional_rows = (integers.notna() & integers.mod(1).ne(0)).any(axis=1)
    record("Nombre de lignes avec une valeur négative", int(negative_rows.sum()))
    record("Nombre de lignes avec un entier fractionnaire",
           int(fractional_rows.sum()))
    # Union logique, et non somme des compteurs : une ligne peut cumuler
    # plusieurs anomalies. Les doublons sont signalés séparément.
    invalid_rows = (missing.any(axis=1) | unreadable_rows | infinite_rows
                    | negative_rows | fractional_rows)

    # 4. Dates : day est le numéro du jour dans l'année (1 à 365/366).
    # Vérifier aussi les années bissextiles et la borne de reproductibilité.
    if {"year", "day"}.issubset(numeric.columns):
        today = pd.Timestamp(reference_date).normalize()
        year, day = numeric["year"], numeric["day"]
        leap = year.mod(4).eq(0) & (
            year.mod(100).ne(0) | year.mod(400).eq(0)
        )
        max_day = pd.Series(np.where(leap, 366, 365), index=data.index)
        valid_year = year.between(2009, today.year) & year.mod(1).eq(0)
        valid_day = day.ge(1) & day.le(max_day) & day.mod(1).eq(0)
        year_start = pd.to_datetime(
            year.where(valid_year).astype("Int64").astype("string"),
            format="%Y", errors="coerce",
        )
        dates = year_start + pd.to_timedelta(day.where(valid_day) - 1, unit="D")
        has_date_fields = ~missing[["year", "day"]].any(axis=1)
        invalid_dates = has_date_fields & ~dates.between("2009-01-01", today)
        record("Nombre de lignes avec une date invalide ou future",
               int(invalid_dates.sum()))
        invalid_rows |= invalid_dates
        valid_dates = dates.where(~invalid_dates)
        record("Début de la période valide", valid_dates.min())
        record("Fin de la période valide", valid_dates.max())
        if "address" in data.columns:
            keys = pd.DataFrame({"address": data["address"], "date": valid_dates})
            eligible = ~missing["address"] & valid_dates.notna()
            record("Nombre de lignes partageant une clé adresse-date",
                   int(keys.loc[eligible].duplicated(keep=False).sum()))

    # 5. Cohérence du graphe : les chemins qui bouclent constituent un
    # sous-ensemble des chemins comptés ; looped ne doit pas dépasser count.
    if {"looped", "count"}.issubset(numeric.columns):
        topology_issue = numeric["looped"].gt(numeric["count"])
        record("Nombre de lignes avec looped > count", int(topology_issue.sum()))
        invalid_rows |= topology_issue
    record("Nombre de lignes distinctes avec une anomalie (hors doublons)",
           int(invalid_rows.sum()))

    if "address" in data.columns:
        record("Nombre d'adresses distinctes",
               data.loc[~missing["address"], "address"].nunique())
    # 6. Statistiques descriptives sur les valeurs convertibles et finies.
    # Les doublons et valeurs négatives restent présents : ceci est un audit,
    # pas un nettoyage. Les quantiles décrivent donc le dataset reçu.
    if not numeric.empty:
        stats = numeric.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
        report["statistiques"] = stats

    _display_report(report, "Vérification du dataset", display_mode)
    return report



def check_label_distribution(data, display_mode="auto"):
    """Affiche la distribution des labels et retourne un rapport séparé."""

    if not data.columns.is_unique:
        raise ValueError("Les noms de colonnes doivent être uniques.")
    if "label" not in data.columns:
        raise ValueError("La colonne label est absente.")
    labels = data["label"]
    # Les chaînes vides ou composées uniquement d'espaces sont aussi absentes.
    missing = labels.isna() | labels.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
    counts = labels.loc[~missing].value_counts()
    distribution = pd.DataFrame({
        "nombre": counts,
        "pourcentage": counts / len(data) * 100 if len(data) else counts * 0,
    })
    report = {
        "Nombre de lignes": len(data),
        "Nombre de labels manquants": int(missing.sum()),
        "Nombre de lignes avec le label white": int(counts.get("white", 0)),
        "Nombre de lignes avec un label autre que white":
            int(counts.sum() - counts.get("white", 0)),
        "distribution_labels": distribution,
    }
    _display_report(report, "Distribution des labels (% de toutes les lignes)",
                    display_mode)
    return report

dataset_report = check_dataset(df, reference_date="2026-10-01")
label_report = check_label_distribution(df)


Le dataset ne présente aucune valeur aberrante ou anomalie, il n'y a pas besoin de filter quoi que ce soit. 
Il contient majoritairement (98.58 %) des valeurs `white`, et la proportion de Ransomware signalés est très faible. On distingue, parmi les principaux, `paduaCryptoWall` avec une fréquence de 0.43 %,  `montrealCryptoLocker` pour 0.32 % et `princetonCerber` avec 0.32%. Le dataset contient 28 labels de ransomware signalé différents.

## 3. Exploration des observations

**Question : quels déséquilibres, associations et évolutions doivent guider la construction des modèles ?**


### 3.1 Déséquilibre des labels

Nous auditons les valeurs et la distribution des labels avant de choisir une tâche de classification. Les lignes `white` sont des observations non signalées dans les sources, et non des exemples dont l'innocence aurait été vérifiée.


In [ ]:
df["is_ransomware"] = df["label"].ne("white").astype("int8")
counts = df["is_ransomware"].value_counts().reindex([0, 1], fill_value=0)
proportions = df["is_ransomware"].value_counts(normalize=True).reindex([0, 1], fill_value=0) * 100

print("\nDistribution des labels binarisés :")
print(f"- white (non signalé) : {counts[0]:,} lignes ({proportions[0]:.2f}%)")
print(f"- ransomware signalé : {counts[1]:,} lignes ({proportions[1]:.2f}%)")

plt.figure(figsize=(7, 4.5))
ax = sns.barplot(
    x=["white (non signalé)", "ransomware signalé"],
    y=counts.values,
    hue=["white (non signalé)", "ransomware signalé"],
    palette=["#2A6F97", "#D1495B"],
    legend=False,
)
ax.set_yscale("log")
ax.set_ylim(top=counts.max() * 5)
ax.set_title("Distribution des labels (échelle logarithmique)", pad=18)
ax.set_ylabel("Nombre d'observations")
for i, value in enumerate(counts.values):
    ax.text(i, value * 1.15, f"{value:,}\n({proportions.iloc[i]:.2f}%)", ha="center")
plt.tight_layout()
plt.show()


* **Constat :** le CSV contient une classe `white` très majoritaire et des observations étiquetées ransomware minoritaires. Ce déséquilibre rend l'accuracy seule peu informative : un classifieur qui prédit toujours `white` serait exact sur environ 98,58 % des lignes sans retrouver aucun positif.
* **Évaluation :** rapporter au minimum Average Precision (résumé du classement sur la courbe précision-rappel, distinct d’une intégration trapézoïdale), précision, rappel, F1, matrice de confusion et nombre de faux positifs par vrai positif. La précision de référence d'un classement aléatoire est proche de la prévalence positive (environ 1,42 % ici).
* **Apprentissage :** comparer un modèle sans pondération à une stratégie de pondération ou de sous-échantillonnage appliquée uniquement aux données d'entraînement. Choisir le seuil et les paramètres sur une validation distincte; ne pas rééquilibrer le test, afin de conserver sa prévalence réelle.


### 3.2 Associations entre les caractéristiques

Nous examinons les associations deux à deux entre les six caractéristiques de graphe et la cible binaire. Pearson décrit une association linéaire; Spearman décrit une association monotone fondée sur les rangs. Ces matrices sont descriptives : une corrélation faible avec la cible ne suffit pas à conclure qu'une variable est inutile, ni qu'un modèle linéaire ne peut pas fonctionner.


In [ ]:
corr_pearson = df[features + ["is_ransomware"]].corr(method="pearson")
corr_spearman = df[features + ["is_ransomware"]].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, matrix, title in zip(
    axes,
    [corr_pearson, corr_spearman],
    ["Pearson", "Spearman"],
):
    sns.heatmap(
        matrix,
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(title)

fig.suptitle("Associations deux à deux : features et cible binaire")
plt.tight_layout()
plt.show()


* **Interprétation :** ces cartes résument seulement des associations deux à deux dans l'échantillon observé. La classe très minoritaire, les relations non linéaires et les interactions entre variables limitent ce qu'on peut conclure à partir d'une corrélation marginale avec la cible.
* Une corrélation entre deux features signale une redondance potentielle, pas automatiquement un défaut du modèle. Les arbres peuvent exploiter des variables corrélées, mais leurs importances peuvent alors être réparties entre elles; une comparaison de modèles et une validation hors période restent nécessaires.

**Constats sur les cartes affichées.** `length` et `count` sont associés à la fois selon Pearson (environ 0,70) et selon Spearman (environ 0,74). À l’inverse, l’association entre `weight` et `neighbors` vaut environ 0,69 selon Pearson mais seulement 0,01 selon Spearman : cet écart invite à examiner les valeurs influentes avant de conclure à une relation générale entre ces variables. Les associations marginales avec la cible restent proches de zéro sur ces cartes ; cela ne contredit pas la possibilité d’interactions exploitables par une forêt. La redondance entre certaines mesures devra être prise en compte dans la lecture des importances.


### 3.3 Distributions et valeurs extrêmes

Les caractéristiques ont des échelles et des distributions différentes. Nous comparons ici leurs distributions transformées pour mieux voir la masse centrale et les queues. Cette visualisation ne sert pas à déclarer automatiquement les valeurs extrêmes invalides; `weight` est strictement positif et sera affiché en `log10`, tandis que les variables non négatives utilisent `log1p`.


In [ ]:
# Échantillon équilibré par classe, uniquement pour comparer les formes des distributions.
# Les proportions de cet échantillon ne représentent pas celles du dataset.
samples = []
for _, group in df.groupby("is_ransomware", observed=True):
    samples.append(group.sample(n=min(25_000, len(group)), random_state=42))
sample_df = pd.concat(samples, ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), features):
    if col == "weight":
        plot_values = np.log10(sample_df[col])
        scale_name = "log10"
    else:
        plot_values = np.log1p(sample_df[col])
        scale_name = "log1p"

    plot_df = sample_df[["is_ransomware"]].copy()
    plot_df["value"] = plot_values
    sns.boxplot(
        data=plot_df,
        x="is_ransomware",
        y="value",
        hue="is_ransomware",
        palette=["#2A6F97", "#D1495B"],
        legend=False,
        showfliers=False,
        ax=ax,
    )
    ax.set_title(f"{scale_name}({col})")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["white", "ransomware"])
    ax.set_xlabel("Label fourni")
    ax.set_ylabel("Valeur transformée")

fig.suptitle("Distributions par classe (échantillon équilibré pour la visualisation)")
plt.tight_layout()
plt.show()


* **Lecture :** l'échantillonnage équilibré rend les distributions des observations minoritaires visibles, mais ne reflète pas la prévalence réelle. Les valeurs transformées facilitent la comparaison de la masse centrale; elles ne remplacent pas l'examen des quantiles et unités originales.
* **Modélisation :** conserver les valeurs extrêmes tant qu'elles ne sont pas identifiées comme erreurs. Comparer les arbres sur les variables brutes aux modèles sensibles à l'échelle sur des transformations ajustées uniquement à partir des données d'entraînement. Les arbres n'exigent pas de normalisation; `log1p` ou une mise à l'échelle pourront être évalués pour les modèles linéaires/SVM.

---

**Constats sur les distributions affichées.** Dans cet échantillon, la distribution centrale des montants des observations positives est plus resserrée et sa médiane est plus basse que celle des `white`, avec un recouvrement important. Un seuil fondé uniquement sur le montant ne peut donc pas être déduit de cette figure. La boîte de `looped` est écrasée sur zéro : les quartiles centraux ne rendent pas visibles toutes les valeurs non nulles, d’autant que les points extrêmes sont masqués par `showfliers=False`. Ce résultat ne permet pas de conclure à l’absence de boucles dans les données.


### 3.4 Relations entre paires de caractéristiques

Le pair-plot présente les distributions individuelles et les relations deux à deux. Au maximum 2 000 observations sont retenues par classe pour conserver une figure lisible. L'échantillon et la transformation `log1p` servent à la visualisation ; l'apprentissage utilisera les variables brutes. La version Plotly est disponible avec `plot_type="plotly"` pour le zoom et le survol.


In [ ]:
def pair_plot(data, n_points=2000, plot_type="seaborn", *, features=None,
              use_log=True, class_mode="binary", diag_kind="hist",
              random_seed=42, height=2.4, save_path=None, show=True):
    """Crée un pair-plot statique ou interactif sans modifier data.

    n_points : maximum de points PAR CLASSE (2000 par défaut).
    plot_type : 'seaborn' pour une figure statique, 'plotly' pour le zoom/survol.
    features : variables à comparer, par défaut les six mesures du graphe.
    use_log : log1p par défaut ; False conserve les unités brutes.
    class_mode : 'binary' (white/autres labels), 'label' (familles originales)
        ou 'none' (sans classes, n_points maximum sur toute la figure).
    diag_kind : 'hist' ou 'kde' pour Seaborn. Diagonale masquée dans Plotly.
    random_seed : graine pour comparer les mêmes points entre les rendus.
    height : taille d'une case en pouces, adaptée en pixels pour Plotly.
    save_path : PNG/PDF/SVG pour Seaborn, HTML autonome pour Plotly.
    show : False crée/exporte la figure sans l'afficher."""

    from numbers import Integral
    from pathlib import Path

    # Valider les options avant les calculs pour donner des erreurs explicites.
    if not isinstance(data, pd.DataFrame):
        raise TypeError("data doit être un DataFrame pandas.")
    if plot_type not in {"seaborn", "plotly"}:
        raise ValueError("plot_type doit être seaborn ou plotly.")
    if random_seed is not None and (not isinstance(random_seed, Integral) or random_seed < 0):
        raise ValueError("random_seed doit être un entier non négatif.")
    if class_mode not in {"binary", "label", "none"}:
        raise ValueError("class_mode doit être binary, label ou none.")
    if diag_kind not in {"hist", "kde"}:
        raise ValueError("diag_kind doit être hist ou kde.")
    if not np.isfinite(height) or height <= 0:
        raise ValueError("height doit être positif et fini.")
    if features is None:
        features = ["length", "weight", "count", "looped", "neighbors", "income"]
    elif isinstance(features, str):
        raise ValueError("features doit être une liste de noms.")
    else:
        features = list(features)
    if len(features) < 2 or len(set(features)) != len(features):
        raise ValueError("Choisir au moins deux variables distinctes.")
    if any(col in {"address", "label", "classe"} for col in features):
        raise ValueError("features doit contenir uniquement des mesures numériques.")
    required = features + (["label"] if class_mode != "none" else [])
    absent = [col for col in required if col not in data.columns]
    if absent:
        raise ValueError(f"Colonnes absentes : {absent}")
    path = Path(save_path) if save_path is not None else None
    if path is not None:
        extensions = {".html"} if plot_type == "plotly" else {".png", ".pdf", ".svg"}
        if path.suffix.lower() not in extensions:
            raise ValueError(f"Extensions acceptées : {sorted(extensions)}")

    # Coordonnées temporaires : aucune conversion ne touche au dataset reçu.
    viz = data[features].apply(pd.to_numeric, errors="coerce")
    eligible = viz.notna().all(axis=1) & ~viz.isin([np.inf, -np.inf]).any(axis=1)
    if use_log:
        eligible &= viz.ge(0).all(axis=1)
    if class_mode != "none":
        labels = data["label"]
        missing = labels.isna() | labels.map(
            lambda value: isinstance(value, str) and not value.strip())
        eligible &= ~missing
        viz["classe"] = (
            np.where(labels.eq("white").fillna(False),
                     "White — non identifié", "Ransomware identifié")
            if class_mode == "binary" else labels.astype("string"))

    # Plotly conserve les unités originales au survol, même sur les axes log.
    if plot_type == "plotly":
        for col in ["address", "date", "label"]:
            if col in data.columns and col not in viz.columns:
                viz[col] = data[col]
        for col in features:
            viz[f"{col}_brut"] = viz[col]
    viz = viz.loc[eligible].copy()
    if viz.empty:
        raise ValueError("Aucune ligne traçable pour les variables/échelle choisies.")

    # Échantillonner AVANT log1p conserve les mêmes points entre les échelles.
    # Les classes plus petites sont gardées en entier, sans suréchantillonnage.
    if class_mode == "none":
        viz = viz.sample(n=min(n_points, len(viz)), random_state=random_seed)
    else:
        viz = pd.concat([
            group.sample(n=min(n_points, len(group)), random_state=random_seed)
            for _, group in viz.groupby("classe", observed=True, sort=True)])
    viz = viz.reset_index(drop=True)
    if use_log:
        viz[features] = np.log1p(viz[features])
    axis_labels = {col: f"log1p({col})" if use_log else col for col in features}
    palette = {"White — non identifié": "#2A6F97",
               "Ransomware identifié": "#D1495B"} if class_mode == "binary" else None
    hue = "classe" if class_mode != "none" else None
    title = "Relations entre les mesures des adresses Bitcoin"
    subtitle = "Échelle log1p" if use_log else "Unités brutes"
    subtitle += f" · {len(viz):,} points"
    if hue:
        subtitle += " · échantillon par classe, proportions non représentatives"
    print(f"Lignes exclues du graphique : {len(data) - int(eligible.sum())}")
    print(f"Points affichés : {len(viz)}")
    if hue:
        _display_report({"effectifs_affiches": viz[hue].value_counts().to_frame("nombre")},
                        "Échantillon du pair-plot")
    corner = True
    if plot_type == "seaborn":
        import matplotlib.pyplot as plt
        import seaborn as sns

        # Style local : les autres figures du notebook conservent leur thème.
        # Nuages rasterisés pour alléger PDF/SVG ; textes et axes vectoriels.
        
        with sns.axes_style("whitegrid"), sns.plotting_context("notebook", font_scale=0.9):
            grid = sns.pairplot(
                viz, vars=features, hue=hue, palette=palette, corner=corner,
                diag_kind=diag_kind, height=height,
                plot_kws={"s": 12, "alpha": 0.3, "linewidth": 0, "rasterized": True},
                diag_kws={"bins": 30, "stat": "density", "common_norm": False,
                          "element": "step", "fill": False}
                if diag_kind == "hist" else {"common_norm": False, "fill": False})
            # Changer les titres des axes, sans renommer les colonnes sources.
            for ax in grid.axes.flat:
                if ax is not None:
                    ax.set_xlabel(axis_labels.get(ax.get_xlabel(), ax.get_xlabel()))
                    ax.set_ylabel(axis_labels.get(ax.get_ylabel(), ax.get_ylabel()))
                    ax.tick_params(labelsize=9)
            grid.figure.suptitle(f"{title}\n{subtitle}", fontsize=13, y=0.99)
            grid.figure.subplots_adjust(top=0.90, bottom=0.08, wspace=0.08, hspace=0.08)
            if path is not None:
                grid.figure.savefig(path, dpi=300, bbox_inches="tight")
            if show:
                plt.show()
            else:
                # Empêcher l'affichage automatique de Matplotlib dans Jupyter.
                # L'objet retourné reste disponible pour l'export/personnalisation.
                plt.close(grid.figure)
        return grid

    import plotly.express as px

    # Matrice interactive ; pas d'histogrammes sur la diagonale Plotly.
    hover = {f"{col}_brut": ":,.3f" for col in features}
    hover.update({col: True for col in ["date", "label"] if col in viz.columns})
    fig = px.scatter_matrix(
        viz, dimensions=features, color=hue, color_discrete_map=palette,
        hover_name="address" if "address" in viz.columns else None,
        hover_data=hover, labels=axis_labels,
        title=f"{title}<br><sup>{subtitle}</sup>",
        height=max(650, int(len(features) * height * 100)),
        width=max(750, int(len(features) * height * 100)))
    fig.update_traces(diagonal_visible=False, showupperhalf=not corner,
                      marker={"size": 4, "opacity": 0.4, "line": {"width": 0}})
    fig.update_layout(template="plotly_white", dragmode="zoom", hovermode="closest",
                      font={"size": 11}, legend={"orientation": "h", "y": -0.12},
                      margin={"l": 80, "r": 40, "t": 100, "b": 120})
    config = {"scrollZoom": True, "displaylogo": False, "responsive": True,
              "toImageButtonOptions": {"format": "png", "filename": "pairplot",
                                       "width": 1800, "height": 1600, "scale": 2}}
    if path is not None:
        fig.write_html(path, include_plotlyjs=True, config=config)
    if show:
        fig.show(config=config)
    return fig


pair_grid = pair_plot(
    df, n_points=2000, plot_type="seaborn",
    features=features, use_log=True, random_seed=seed,
)


Les six caractéristiques permettent d'examiner des relations qui ne sont pas résumées par une corrélation unique. Les zones de recouvrement entre labels empêchent d'assimiler une région du graphique à une preuve de malveillance. 
Les observations positives sont rendues visibles grâce à l'échantillonnage par classe : la proportion de points rouges dans la figure ne correspond pas à leur fréquence dans le fichier. `log1p(income)` porte ici sur les satoshis. 

**Constats sur les projections affichées :** les deux labels se recouvrent dans de nombreux nuages. Le couple `count`–`looped` présente à la fois une concentration à `looped = 0` et des observations proches de la diagonale, cohérentes avec le fait que `looped` est un sous-ensemble de `count`. Les concentrations sur certaines valeurs de `length` et de `neighbors` reflètent des caractéristiques discrètes ; elles ne désignent pas à elles seules des familles. Ces observations motivent l’étude des interactions et du ratio `loop_ratio`, sans établir leur gain prédictif.


### 3.5 Répartition temporelle et réapparition des adresses

L'évolution des effectifs prépare le choix des fenêtres chronologiques. Les réapparitions expliquent pourquoi les observations d'une même adresse doivent rester groupées dans l'expérience initiale.


In [ ]:
year_counts = pd.crosstab(df["year"], df["is_ransomware"]).reindex(columns=[0, 1], fill_value=0)
year_counts.columns = ["white (non signalé)", "ransomware signalé"]
address_counts = df["address"].value_counts()

fig, ax = plt.subplots(figsize=(9, 4.5))
year_counts.plot.bar(ax=ax, logy=True, color=["#2A6F97", "#D1495B"])
ax.set(xlabel="Année", ylabel="Nombre d'observations (échelle logarithmique)",
       title="Disponibilité des observations par période et label")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
display(year_counts)
display(pd.Series({
    "Adresses distinctes": address_counts.size,
    "Adresses observées plusieurs fois": int(address_counts.gt(1).sum()),
    "Observations au-delà de la première par adresse": int(address_counts.sub(1).sum()),
}, name="Effectif").to_frame())


Les effectifs positifs varient fortement selon l'année. Dans le fichier actuel, 2018 ne contient que trois observations étiquetées ransomware : un test limité à cette année fournirait une estimation très fragile du rappel. Les fenêtres devront donc être choisies selon la chronologie et les effectifs, puis fixées avant les comparaisons de modèles.

Une adresse récurrente fournit plusieurs observations liées. Une séparation aléatoire par ligne risquerait de placer la même adresse dans l'entraînement et dans le test. La séparation par adresse évite ce recouvrement, sans garantir pour autant une séparation temporelle ou une indépendance de tous les comportements de campagne.

**Limite de lecture des volumes.** Les effectifs `white` sont proches d’une année à l’autre entre 2012 et 2017, tandis que les effectifs positifs varient davantage. Ces volumes dépendent de la construction du fichier et de ses sources : ils ne doivent pas être interprétés directement comme le nombre réel de rançons ou comme l’évolution de la criminalité sur Bitcoin.

## 4. Construction et sélection des caractéristiques

### 4.1 Référence, unité et transformations

Les six caractéristiques originales restent la référence. `address` identifie les groupes ; `year`, `day` et `date` servent à situer les observations ; `label` et `is_ransomware` définissent la cible. Ces colonnes ne sont pas incluses dans les entrées de la forêt.

La normalisation change l'échelle des variables, par exemple en les centrant et en les divisant par leur écart-type. Elle est utile pour les méthodes sensibles aux distances ou à l'échelle, telles que les SVM. Les arbres choisissent des seuils variable par variable et n'exigent généralement pas cette normalisation. Les transformations logarithmiques restent utiles pour les figures ; un gain prédictif éventuel doit être testé séparément. Les quantités apprises lors d'une imputation ou d'une normalisation seront ajustées uniquement sur l'entraînement.

### 4.2 Deux ratios à évaluer

| Caractéristique | Définition | Hypothèse à tester |
|---|---|---|
| `loop_ratio` | `looped / count`, lorsque `count > 0` | La part des transactions de départ atteignant l'adresse par plusieurs chemins apporte-t-elle une information utile ? |
| `income_per_neighbor` | `income / neighbors`, lorsque `neighbors > 0` | Le montant moyen reçu par transaction entrante distingue-t-il certains comportements ? |

Ces ratios réexpriment les six mesures ; ils n'ajoutent pas de source externe. Un dénominateur nul produit une valeur absente, pas un zéro supposé. Ils ne sont pas ajoutés automatiquement à la forêt de référence.


In [ ]:
def make_features(data):
    """Construit deux ratios interprétables dans une copie indépendante.
    Un dénominateur nul donne une valeur absente, jamais un zéro inventé.
    """
    result = data[features].copy()
    result["loop_ratio"] = data["looped"].div(data["count"].where(data["count"].gt(0)))
    result["income_per_neighbor"] = data["income"].div(
        data["neighbors"].where(data["neighbors"].gt(0))
    )
    return result

feature_data = make_features(df)
engineered_features = ["loop_ratio", "income_per_neighbor"]
display(feature_data[engineered_features].describe(percentiles=[0.01, 0.5, 0.99]).T)
display(feature_data[engineered_features].isna().sum().rename("missing_values").to_frame())

Ces statistiques vérifient que les définitions produisent des quantités interprétables et recensent les cas non définis. Elles ne démontrent pas une amélioration de la détection. L'expérience à conduire compare, sur les mêmes partitions et avec le même budget de réglage, la forêt sur les six caractéristiques puis sur les six caractéristiques complétées par chaque ratio et par les deux ratios.

Les deux ratios sont définis pour les 2 916 697 observations et ne créent aucune valeur manquante dans cette version du CSV. `loop_ratio` est compris entre 0 et 1, conformément à `looped <= count`. Le montant moyen par transaction entrante reste très asymétrique ; sa moyenne ne suffit pas à représenter une observation typique.

### 4.3 Information historique — extension à préciser

Le nombre d'observations antérieures d'une adresse, un montant cumulé passé et une variation récente d'activité pourraient compléter la représentation. Ils doivent être calculés après tri chronologique, uniquement avec le passé, et leur signification dépend de la couverture du CSV : une adresse absente du fichier n'est pas nécessairement inactive sur la blockchain. Le fichier ne contient pas les transactions brutes nécessaires pour reconstruire de nouvelles propriétés du graphe. Ces variables historiques ne sont donc pas produites artificiellement à ce stade.


## 5. Apprentissage : première forêt aléatoire

L'objectif est d'estimer, à partir de six caractéristiques d'activité, si une observation quotidienne d'adresse présente le label `white` (non signalée) ou un label de rançongiciel connu. Cette étape relève de l'**apprentissage supervisé** : les caractéristiques constituent les entrées et les labels disponibles servent de cible pendant l'entraînement.

Le label `white` correspond à l'absence de signalement dans les sources, les performances mesurent donc la concordance avec les annotations disponibles.

La forêt aléatoire agrège plusieurs arbres de décision. Chaque arbre est entraîné sur un échantillon tiré avec remise ; à chaque nœud, les coupures sont recherchées parmi un sous-ensemble aléatoire des variables. Ces mécanismes diversifient les arbres et réduisent la variance de l'ensemble. L'implémentation retenue moyenne leurs probabilités de classe, puis sélectionne la classe la plus probable.

Soit $T=\{(x^i,y^i)\}_{i=1}^N$ le jeu d'entraînement et $B$ le nombre d'arbres. Chaque arbre $\varphi^b$ est entraîné sur un échantillon bootstrap $T^b$. L'agrégation par vote majoritaire s'écrit :

$$\varphi_B(x)=\arg\max_{y\in\{0,1\}}\sum_{b=1}^{B}\mathbb{I}\!\left(\varphi^b(x)=y\right).$$

L'indicatrice $\mathbb{I}$ vaut 1 lorsque l'arbre prédit la classe $y$, et 0 sinon. Cette formule décrit le vote des classes. Pour l'implémentation utilisée, l'agrégation porte sur les probabilités estimées par les arbres :

$$\widehat{p}(y\mid x)=\frac{1}{B}\sum_{b=1}^{B}\widehat{p}_b(y\mid x),\qquad
\widehat{y}(x)=\arg\max_{y\in\{0,1\}}\widehat{p}(y\mid x).$$

Ces deux règles d'agrégation peuvent produire des prédictions différentes lorsque les feuilles des arbres ne sont pas pures.


### 5.1 Justification des modèles

Random Forest et XGBoost sont adaptés à une classification supervisée sur variables tabulaires et à des relations non linéaires. Random Forest combine des arbres diversifiés par bootstrap et sélection aléatoire de variables. XGBoost ajoute progressivement des arbres pour réduire une fonction de perte, avec des mécanismes de régularisation. Leur pertinence ne garantit pas qu'ils se séparent correctement sur les labels disponibles.

### 5.2 Définition des partitions et expérience de référence

Une **partition** est une répartition des observations entre plusieurs sous-ensembles ayant des rôles définis. L'**entraînement** sert à apprendre ; la **validation** sert à choisir les hyperparamètres et le seuil ; le **test final** sert à évaluer la procédure retenue sans guider ces choix. Un **hyperparamètre** est un réglage choisi avant l'apprentissage, comme la profondeur maximale des arbres.

L'expérience déjà réalisée réserve environ 20 % des adresses au test et entraîne la forêt sur les autres. Toutes les observations d'une même adresse restent du même côté. Le pourcentage de lignes peut différer de 20 %, puisque les adresses n'ont pas toutes le même nombre d'observations. Ce test mesure la généralisation à des adresses non vues, et non la prédiction d'un futur non observé.

Les entrées sont `length`, `weight`, `count`, `looped`, `neighbors` et `income` en satoshis. Les contrôles vérifient que chaque jeu contient les deux classes et qu'aucune adresse n'est partagée.


In [ ]:
seed = 45
test_size = 0.20

features = features.copy()
required = [*features, "address", "label"]

X = data[features].to_numpy(dtype=float, copy=True)
y = data["label"].ne("white").to_numpy(dtype=int)
groups = data["address"].to_numpy(dtype=str)

split = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
train_indices, test_indices = next(split.split(X, y, groups=groups))
Xtrain, Xtest = X[train_indices], X[test_indices]
ytrain, ytest = y[train_indices], y[test_indices]

# Vérifier les deux classes et l’absence d’adresse commune entre les jeux.
if set(np.unique(ytrain)) != {0, 1} or set(np.unique(ytest)) != {0, 1}:
    raise ValueError("Les deux classes doivent être présentes dans train et test.")
shared_addresses = np.intersect1d(groups[train_indices], groups[test_indices])
if shared_addresses.size:
    raise ValueError("Des adresses sont communes aux jeux d’entraînement et de test.")

split_report = pd.DataFrame({
    "lignes": [len(ytrain), len(ytest)],
    "adresses": [pd.Series(groups[train_indices]).nunique(),
                 pd.Series(groups[test_indices]).nunique()],
    "positifs": [int(ytrain.sum()), int(ytest.sum())],
    "positifs (%)": [100 * ytrain.mean(), 100 * ytest.mean()],
}, index=["Entraînement", "Test"])
display(split_report.round(3))


### 5.3 Apprentissage de la forêt

Le modèle comprend 101 arbres de décision. Les coupures maximisent la réduction d'entropie, c'est-à-dire l'amélioration de la séparation des classes au sein des nœuds. Avec six variables explicatives, le réglage `max_features='sqrt'` conduit à examiner normalement deux variables candidates à chaque coupure ; la recherche peut être étendue si aucune coupure valide n'est trouvée.

Les arbres sont entraînés par bootstrap, sans profondeur maximale imposée. Cette configuration permet des partitions détaillées, mais peut aussi apprendre des particularités du jeu d'entraînement. La comparaison des performances d'entraînement et de test est donc nécessaire. Les classes ne sont pas pondérées dans cette première configuration ; la pondération constitue une variante à évaluer sur validation.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=101,
    criterion='entropy',
    max_features='sqrt',
    bootstrap=True,
    class_weight=None,
    random_state=seed,
    n_jobs=2,  # Deux tâches de calcul en parallèle.
)
rf.fit(Xtrain, ytrain)

print("Training score: %g" % rf.score(Xtrain, ytrain))
print("Generalization score: %g" % rf.score(Xtest, ytest))
print("Generalization error: %g" % (1. - rf.score(Xtest, ytest)))
print("Always-white score: %g" % np.mean(ytest == 0))


**Interprétation des résultats.**

**Résultats observés :** L'exactitude atteint **99,56 % à l'entraînement** et **98,46 % au test**, soit un écart d'environ **1,10 point**. Cet écart est compatible avec un ajustement aux particularités de l'entraînement ; il ne permet pas, à lui seul, d'en déterminer la cause ni d'en estimer la variabilité.

L'erreur de test est de **1,54 %**. La stratégie de référence consistant à attribuer systématiquement `white` atteint pourtant **98,6413 % d'exactitude**, soit environ **0,185 point de plus** que la forêt. À coût identique pour toute erreur, le modèle n'améliore donc pas ce critère global. Cette comparaison ne signifie pas que ses alertes sont inutiles : la stratégie de référence ne retrouve aucun positif. L'intérêt opérationnel dépend du rappel obtenu et du coût des fausses alertes, évalués dans la section suivante.


### 5.4 Performance de détection et analyse des erreurs

Les tableaux et graphiques suivants évaluent la détection par classe et le compromis entre rappel et précision.


In [ ]:
ypredict = rf.predict(Xtest)

# Repérer la colonne de la classe positive 1 (ransomware signalé).
positive_column = int(np.flatnonzero(rf.classes_ == 1)[0])
yproba = rf.predict_proba(Xtest)[:, positive_column]
misclass = np.not_equal(ypredict, ytest)

cm = confusion_matrix(ytest, ypredict, labels=[0, 1])
TN, FP, FN, TP = cm.ravel()
metrics = pd.Series({
    "Accuracy": rf.score(Xtest, ytest),
    "Précision (classe 1)": precision_score(ytest, ypredict, zero_division=0),
    "Rappel (classe 1)": recall_score(ytest, ypredict, zero_division=0),
    "F1 (classe 1)": f1_score(ytest, ypredict, zero_division=0),
    "Average Precision": average_precision_score(ytest, yproba),
    "Prévalence positive du test": ytest.mean(),
    "FP / TP": FP / TP if TP else np.nan,
    "TN": TN, "FP": FP, "FN": FN, "TP": TP,
    "Nombre d'erreurs": int(misclass.sum()),
}, name="Résultat")
display(metrics.to_frame().round(4))


report = classification_report(
    ytest, ypredict, labels=[0, 1],
    target_names=["white (non signalé)", "Ransomware signalé"],
    output_dict=True, zero_division=0,
)

report_table = pd.DataFrame({
    name: values for name, values in report.items() if name != "accuracy"
}).T.rename(index={
    "macro avg": "Moyenne non pondérée",
    "weighted avg": "Moyenne pondérée",
}, columns={"precision": "Précision", "recall": "Rappel",
            "f1-score": "F1", "support": "Effectif réel"})
display(report_table.style.format({
    "Précision": "{:.4f}", "Rappel": "{:.4f}", "F1": "{:.4f}",
    "Effectif réel": "{:,.0f}",
}))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=["white", "Ransomware"]).plot(
    ax=axes[0], cmap="Blues", values_format="d", colorbar=False,
)
axes[0].set_title("Confusion matrix — test")
axes[0].set_xlabel("Label prédit")
axes[0].set_ylabel("Label fourni")
precision, recall, thresholds = precision_recall_curve(ytest, yproba)
axes[1].step(recall, precision, where="post", label="Random Forest")
axes[1].axhline(ytest.mean(), color="gray", linestyle="--", label="Prévalence du test")
axes[1].scatter(metrics["Rappel (classe 1)"], metrics["Précision (classe 1)"],
                color="red", s=35, label="rf.predict")
axes[1].set(xlabel="Rappel", ylabel="Précision", title="Courbe précision-rappel",
            xlim=(0, 1), ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()


**Interprétation des résultats**


La performance doit être examinée par classe, compte tenu de la forte majorité des observations `white`. Les indicateurs sont calculés relativement aux annotations disponibles : une fausse alerte désigne ici une observation `white` signalée par le modèle, sans établir son innocuité réelle.

| Indicateur | Interprétation |
|---|---|
| Précision de la classe ransomware | Proportion des alertes correspondant à un label ransomware. |
| Rappel de la classe ransomware | Proportion des positifs connus retrouvés par le modèle. |
| F1 de la classe ransomware | Moyenne harmonique de sa précision et de son rappel. |
| Average Precision (AP) | Résumé du classement des observations sur la courbe précision-rappel, pour plusieurs seuils. |
| FP/TP | Nombre de fausses alertes par positif retrouvé ; non défini si aucun positif n'est retrouvé. |
| Support | Nombre d'observations portant réellement le label considéré dans le test. |

**Lecture des moyennes.** Le tableau présente une précision, un rappel et un F1 pour chaque classe. La ligne **moyenne non pondérée** calcule séparément la moyenne des deux précisions, celle des deux rappels et celle des deux F1. Chaque classe y a le même poids, indépendamment de son effectif. Le support affiché sur cette ligne est l'effectif total du test, et non une moyenne des effectifs. Le F1 moyen n'est pas nécessairement la moyenne harmonique de la précision moyenne et du rappel moyen. La **moyenne pondérée** quant à elle pondère chaque indicateur par les effectifs des classes. Elle est donc dominée par les observations `white`. 

**Résultats de référence (`seed = 45`).** Sur 582 826 observations, dont 7 919 positives, la forêt retrouve **657 positifs** et en manque **7 262**. Elle produit **1 734 fausses alertes** pour **2 391 alertes** : la précision est d'environ **27,48 %**, le rappel de **8,30 %** et le F1 de **0,1274**. Environ **72,52 %** des alertes concernent donc des observations `white` et **91,70 %** des positifs connus restent non signalés. Le ratio FP/TP est d'environ **2,64**.

L'Average Precision est de **0,1256**, pour une prévalence positive de référence d'environ **1,36 %**. Le classement contient un signal, mais ce score ne signifie pas que 12,56 % des alertes sont fiables à tout seuil. La moyenne pondérée est dominée par `white` ; le rappel moyen non pondéré est proche de **54 %**, ce qui révèle le déséquilibre entre les deux classes. La configuration actuelle ne suffit pas à retrouver la majorité des positifs connus.

La courbe doit être lue comme un ensemble de compromis : accroître le rappel peut augmenter sensiblement la charge d'investigation. La décision suivante consiste à mesurer ce compromis sur validation, puis à choisir un objectif opérationnel. Les valeurs ci-dessus correspondent aux sorties enregistrées ; toute nouvelle exécution modifiant données ou paramètres exige d'actualiser cette interprétation.

La matrice de confusion présente les labels fournis en lignes et les prédictions en colonnes. Le point sur la courbe précision-rappel correspond à la décision par défaut du modèle. Un autre seuil doit être choisi sur un jeu de validation, selon un objectif explicite de rappel et de charge d'investigation.

**Couverture des familles.** La présence des deux classes 0 et 1 dans chaque jeu ne garantit pas que chaque famille positive soit représentée dans l’entraînement. Le diagnostic par famille devra distinguer les familles présentes dans le train de celles qui n’y apparaissent pas, avant de qualifier une performance de détection de famille connue.


## 6. Amélioration de Random Forest : seuil et hypothèses

**Question : les limites observées viennent-elles du seuil, du poids des classes, de la complexité des arbres ou des caractéristiques disponibles ?** 

### 6.1 Distinguer classement et décision

Le score classe les observations selon le modèle. Le **seuil de décision** est la valeur au-delà de laquelle une observation est signalée. Modifier ce seuil change les alertes et leur précision/rappel ; cela ne réentraîne pas le modèle et ne change pas son Average Precision. Le seuil doit être choisi sur validation, selon une précision minimale ou un budget d'alertes convenu.


In [ ]:

from sklearn.base import clone


def check_group_split(y, groups, partitions):
    """Contrôle la présence des classes, les indices et l'absence d'adresse commune."""
    if len(y) != len(groups):
        raise ValueError("La cible et les groupes doivent avoir la même longueur.")
    used_indices = set()
    used_groups = set()
    for name, indices in partitions.items():
        indices = np.asarray(indices)
        if indices.ndim != 1 or not np.issubdtype(indices.dtype, np.integer):
            raise ValueError(f"Indices invalides dans {name}.")
        if not len(indices) or (indices < 0).any() or (indices >= len(y)).any():
            raise ValueError(f"Indices vides ou hors limites dans {name}.")
        current_indices = set(indices.tolist())
        if len(current_indices) != len(indices) or used_indices.intersection(current_indices):
            raise ValueError("Des observations sont répétées entre les sous-ensembles.")
        if set(np.unique(y[indices])) != {0, 1}:
            raise ValueError(f"Les deux classes doivent être présentes dans {name}.")
        current_groups = set(groups[indices])
        if used_groups.intersection(current_groups):
            raise ValueError("Des adresses sont communes aux sous-ensembles.")
        used_groups.update(current_groups)
        used_indices.update(current_indices)

def partition_summary(y, groups, partitions):
    """Résume les effectifs sans arrondir les proportions avant l'affichage."""
    return pd.DataFrame({
        name: {
            "rows": len(indices),
            "addresses": pd.Series(groups[indices]).nunique(),
            "positives": int(y[indices].sum()),
            "positive_rate_pct": 100 * y[indices].mean(),
        }
        for name, indices in partitions.items()
    }).T


def positive_scores(model, X):
    """Sélectionne le score de la classe 1 dans l'ordre déclaré par le modèle."""
    positions = np.flatnonzero(model.classes_ == 1)
    if len(positions) != 1:
        raise ValueError("Le modèle ne contient pas exactement une classe positive 1.")
    return model.predict_proba(X)[:, int(positions[0])]


def threshold_table(y_true, scores):
    """Compare les seuils sur VALIDATION ; ne choisit aucun objectif métier.

    Les comptages sont calculés sur tous les seuils distincts, sans tableau
    de prédictions de taille n_observations × n_seuils.
    """

    y_true = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    if y_true.ndim != 1 or scores.ndim != 1 or len(y_true) != len(scores):
        raise ValueError("La cible et les scores doivent être deux vecteurs de même taille.")
    if set(np.unique(y_true)) != {0, 1} or not np.isfinite(scores).all():
        raise ValueError("Les deux classes et des scores finis sont requis.")
    if ((scores < 0) | (scores > 1)).any():
        raise ValueError("Les scores doivent appartenir à [0, 1].")
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    tp = np.rint(recall[:-1] * y_true.sum()).astype(int)
    order = np.sort(scores)
    alerts = len(scores) - np.searchsorted(order, thresholds, side="left")
    fp = alerts - tp
    total = precision[:-1] + recall[:-1]
    f1 = np.divide(2 * precision[:-1] * recall[:-1], total,
                   out=np.zeros_like(total), where=total > 0)
    table = pd.DataFrame({
        "threshold": thresholds, "precision": precision[:-1],
        "recall": recall[:-1], "f1": f1, "alerts": alerts,
        "true_positives": tp, "false_positives": fp,
        "fp_per_tp": np.divide(fp, tp, out=np.full(len(tp), np.nan), where=tp > 0),
    })
    return table


def plot_threshold_tradeoff(table):
    """Affiche précision/rappel et charge d'alerte selon le seuil."""

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(table["threshold"], table["precision"], label="Précision")
    axes[0].plot(table["threshold"], table["recall"], label="Rappel")
    axes[0].set(xlabel="Seuil", ylabel="Score", ylim=(0, 1.02),
                title="Compromis sur validation")
    axes[0].legend()
    axes[1].plot(table["threshold"], table["alerts"])
    axes[1].set(xlabel="Seuil", ylabel="Nombre d'observations signalées",
                title="Charge de signalement sur validation")
    fig.tight_layout()
    return fig


validation_split = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed + 1)
fit_positions, validation_positions = next(
    validation_split.split(Xtrain, ytrain, groups=groups[train_indices])
)
fit_indices = train_indices[fit_positions]
validation_indices = train_indices[validation_positions]
check_group_split(y, groups, {
    "fit": fit_indices, "validation": validation_indices, "reference_test": test_indices,
})
display(partition_summary(y, groups, {
    "fit": fit_indices, "validation": validation_indices, "reference_test": test_indices,
}).round(3))

validation_rf = clone(rf)
validation_rf.fit(X[fit_indices], y[fit_indices])
validation_scores = positive_scores(validation_rf, X[validation_indices])
threshold_results = threshold_table(y[validation_indices], validation_scores)
threshold_figure = plot_threshold_tradeoff(threshold_results)
plt.show()



**Analyse à compléter après l'expérience.** La figure, lorsqu'elle est produite, mettra en regard précision, rappel et nombre d'observations signalées pour les mêmes scores de validation. Il faudra indiquer le seuil retenu, la contrainte opérationnelle, le rappel obtenu et le nombre d'alertes correspondant. Un score élevé sur validation ne permettra pas de conclure directement sur la population Bitcoin entière ni sur une période future.

Aucun seuil « optimal » n'est annoncé ici : l'expérience n'a pas encore été exécutée et la capacité d'investigation n'a pas été définie. Si l'objectif est un nombre d'adresses par jour, il faudra regrouper les scores au niveau adresse/journée et évaluer les premières adresses du classement quotidien, plutôt que compter plusieurs fois la même adresse.

### 6.2 Variantes contrôlées à comparer

| Expérience | Hypothèse | Condition de comparaison |
|---|---|---|
| Même forêt, seuil adapté | Le score peut être plus utile que la décision par défaut | Mêmes scores ; seuil choisi sur validation |
| Pondération de la classe positive | Le déséquilibre peut limiter la détection | Poids ajustés sur entraînement uniquement |
| Feuilles plus grandes | Certaines décisions apprennent des cas trop particuliers | Mêmes partitions et variables |
| Profondeur limitée | Une complexité moindre peut améliorer la généralisation | Comparer train et validation, pas seulement train |
| Ratios de la partie 4 | Une autre représentation peut aider les coupures | Même algorithme et même budget de sélection |
| Sous-échantillonnage des `white` | Réduire le coût et modifier l'équilibre d'apprentissage | Validation et test conservés à leur prévalence observée |

Les modifications isolées servent à comprendre les effets ; leurs combinaisons seront ensuite comparées car les réglages peuvent interagir. Une pondération modifie l'apprentissage et peut aussi modifier la calibration des scores. Un score n'est pas automatiquement une probabilité de risque fiable.


## 7. Réglage des hyperparamètres et variabilité

**Question : les gains observés sont-ils stables et justifient-ils le coût du modèle ?** Une exactitude d'entraînement élevée accompagnée d'une performance de validation plus faible peut signaler du **surapprentissage (overfitting)**, c'est-à-dire l'apprentissage de particularités peu transférables.

### 7.1 Espace de recherche proposé

| Hyperparamètre | Valeurs de départ | Effet étudié |
|---|---|---|
| `min_samples_leaf` | 1, 5, 20, 100 | Régulariser les décisions locales |
| `max_depth` | 10, 20, 30, sans limite | Limiter la complexité |
| `max_features` | 2, 3, 4, 6 pour la référence à six variables | Diversifier les arbres |
| `class_weight` | Aucune, équilibrée, quelques pondérations positives | Modifier le coût relatif des classes |
| `max_samples` | 0,5 ; 0,8 ; totalité du train | Diversité et coût d'entraînement |
| `n_estimators` | Environ 100, puis 300 ou 500 pour les finalistes | Vérifier la stabilisation |
| `criterion` | Entropie, puis Gini en vérification | Sensibilité au critère de coupure |

Ces valeurs définissent des hypothèses de recherche, pas une grille exhaustive à lancer. Une recherche aléatoire limitée sur développement, puis une confirmation des meilleurs candidats sur davantage de données, permettra de maîtriser le coût. Une optimisation bayésienne pourra ensuite allouer les essais les plus coûteux selon les résultats précédents.

Le critère principal proposé est l'Average Precision de validation, complétée par le rappel à précision minimale ou par la qualité des premières alertes selon l'objectif retenu. Le F1 n'est pas retenu automatiquement comme objectif métier. Les résultats devront inclure moyenne, dispersion, temps d'entraînement et paramètres. Les durées et scores ne sont pas inventés tant que ces essais ne sont pas exécutés.

### 7.2 Validation par partitions répétées

Le code existant ci-dessous répète les partitions à l'intérieur du train de référence. Il reste désactivé par défaut : vingt essais correspondent à vingt forêts supplémentaires. Il permet d'estimer la dispersion d'une configuration fixée ; il ne constitue pas, à lui seul, une recherche d'hyperparamètres ou une confirmation temporelle.

Les validations se recoupent éventuellement : il ne s'agit pas de vingt sous-ensembles disjoints et l'écart-type ne constitue pas un intervalle de confiance fondé sur vingt essais indépendants. Les paramètres et le seuil doivent être choisis sans utiliser le test de référence.


In [ ]:
RUN_REPEATED_VALIDATION = False
nb_trials = 20
score = []
ap_score = []

if RUN_REPEATED_VALIDATION:
    groups_train = groups[train_indices]
    validation = GroupShuffleSplit(n_splits=nb_trials, test_size=0.20, random_state=123)
    for i, (learn, valid) in enumerate(validation.split(Xtrain, ytrain, groups_train)):
        if len(np.unique(ytrain[learn])) != 2 or len(np.unique(ytrain[valid])) != 2:
            raise ValueError(f"Une classe est absente de l'essai {i + 1}.")
        # Un nom distinct évite d'écraser rf, la forêt évaluée sur le test final.
        trial_rf = RandomForestClassifier(
            n_estimators=101, criterion='entropy', random_state=42 + i, n_jobs=2,
        )
        trial_rf.fit(Xtrain[learn], ytrain[learn])
        score += [trial_rf.score(Xtrain[valid], ytrain[valid])]
        ap_score += [average_precision_score(
            ytrain[valid], trial_rf.predict_proba(Xtrain[valid])[:, 1],
        )]
        print('*', end='', flush=True)
        del trial_rf
    print(" done!")
    print("Average generalization score:", np.mean(score))
    print("Standard deviation:", np.std(score))
    print("Average validation AP:", np.mean(ap_score))
    print("AP standard deviation:", np.std(ap_score))
else:
    print("Répétitions non exécutées. Activer RUN_REPEATED_VALIDATION si nécessaire.")


**Lecture des répétitions.** Une moyenne élevée accompagnée d'une forte dispersion signalerait une sensibilité au choix des adresses. Un gain faible devant cette dispersion ne suffirait pas à recommander une configuration. La cellule est actuellement désactivée ; aucune stabilité empirique n'est affirmée.

### 7.3 Confirmation et traçabilité — à réaliser

Pour les candidats retenus, utiliser les mêmes partitions et un budget comparable. Enregistrer pour chaque essai les données utilisées, la graine, les paramètres, les versions des bibliothèques, les métriques et les temps. Une validation externe de la procédure de sélection ou de nouvelles données seront nécessaires pour une estimation indépendante des choix. Une modification de graine ne suffit pas à effacer les décisions déjà prises en consultant le fichier.


## 8. Comparaison avec XGBoost — protocole à exécuter

**Question : un ensemble d'arbres appris séquentiellement améliore-t-il le compromis entre détection et fausses alertes ?** XGBoost corrige progressivement une fonction de perte, tandis que Random Forest agrège des arbres diversifiés appris par bootstrap. La comparaison ne suppose pas que l'un soit supérieur.

### 8.1 Comparaison équitable

Quatre configurations seront comparées : RF initiale, RF retenue après validation, XGBoost initial et XGBoost retenu après validation. Les observations, variables, partitions, objectifs et budgets de recherche seront communs. Les six variables de référence seront d'abord comparées seules ; les variables dérivées constitueront une expérience distincte pour ne pas attribuer au modèle un gain dû aux entrées.

### 8.2 Paramètres prioritaires

| Paramètres | Question à traiter |
|---|---|
| `max_depth`, `min_child_weight` | Quel niveau de complexité généralise ? |
| `learning_rate`, `n_estimators` | Quel compromis entre progression et nombre d'arbres ? |
| `subsample`, `colsample_bytree` | Quelle diversification réduit la sensibilité aux observations ? |
| `reg_alpha`, `reg_lambda`, `gamma` | Quelle régularisation limite les coupures peu utiles ? |
| `scale_pos_weight` | Une pondération des positifs aide-t-elle sur validation ? |
| Arrêt anticipé | Quand cesser l'ajout d'arbres selon une validation distincte ? |

### 8.3 Résultats attendus dans le rapport

Présenter l'AP, la précision et le rappel à objectif opérationnel comparable, les effectifs TP/FP/FN/TN, la variabilité et les coûts. Un modèle moins précis peut être intéressant s'il retrouve davantage de cas à charge acceptable ; un modèle plus complexe doit justifier son coût. Les scores XGBoost restent à mesurer : aucun résultat ni graphique de comparaison n'est produit à ce stade.


## 9. Explication des décisions et diagnostic des erreurs

### 9.1 Importance des caractéristiques de la forêt de référence

Les importances représentent la part normalisée de réduction d'impureté attribuée à chaque variable lors de l'entraînement. Elles totalisent 1. Une importance de 0,55 signifie qu'environ 55 % de cette réduction est attribuée à la variable ; elle ne signifie ni 55 % d'exactitude, ni 55 % de cas détectés.


In [ ]:
importances = rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 6))
plt.title("Feature importances — Random Forest")
plt.bar(range(X.shape[1]), importances[indices], color="r",
        yerr=std[indices], align="center", capsize=4)
plt.xticks(range(X.shape[1]), np.array(features)[indices], rotation=35, ha="right")
plt.xlim([-1, X.shape[1]])
plt.ylabel("Diminution moyenne d'impureté normalisée")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "variable": np.array(features)[indices],
    "importance": importances[indices],
    "std_entre_arbres": std[indices],
}).round(5))


**Interprétation de la figure de référence.** Le montant reçu `income` représente **55,52 %** de la réduction d'impureté normalisée, suivi de `weight` (**19,32 %**) et `count` (**10,99 %**). Ces trois caractéristiques concentrent environ **85,82 %** de l'importance. `length` représente **7,75 %**, `looped` **3,34 %** et `neighbors` **3,09 %**. La forêt utilise principalement le montant et certaines propriétés d'agrégation des flux pour partitionner l'échantillon.

Les barres représentent l'écart-type des importances entre arbres. Pour `income`, il est de **0,01495** autour d'une importance de **0,55516** : sa prépondérance est relativement stable à l'intérieur de cette forêt. Cette stabilité ne garantit pas qu'elle se reproduise sur une autre période. Ces chiffres sont issus de la même référence en satoshis que les scores de la partie 5 ; ils devront être actualisés après tout nouvel entraînement.

Ces importances n'indiquent ni le sens de l'effet ni une causalité. Les variables à nombreuses valeurs distinctes peuvent être favorisées et les variables corrélées peuvent se partager l'importance. Une faible valeur ne justifie pas automatiquement une suppression. Les performances de la référence restent limitées malgré la stabilité de cette hiérarchie.

### 9.2 Importance par permutation, ablations et explications locales — à réaliser

Sur validation, une importance par permutation mesurera la dégradation du score lorsque l'information d'une variable est perturbée. Des variables corrélées peuvent masquer mutuellement leur contribution : les résultats devront être rapprochés des corrélations de la partie 3. Une **ablation** consiste à retirer une variable ou un groupe de variables puis à réentraîner dans les mêmes conditions ; l'expérience sans `income` permettra notamment d'examiner la dépendance au montant.

SHAP ou LIME pourront expliquer quelques vrais positifs, faux positifs et faux négatifs. Une explication décrit le comportement du modèle, pas la cause d'une activité malveillante. Les cas seront choisis pour couvrir plusieurs familles, montants et périodes, sans présenter une sélection de quelques exemples comme représentative de toutes les erreurs.

### 9.3 Diagnostic par population — à réaliser

Mesurer le rappel de chaque famille, les performances par période et par tranche de montant. Examiner aussi les combinaisons de six caractéristiques portant des labels opposés : elles peuvent signaler une information insuffisante ou des annotations ambiguës, qu'un réglage plus complexe ne résoudra pas nécessairement. Le rapport principal présentera les tendances agrégées ; les adresses individuelles serviront d'annexe d'investigation si elles sont utiles.


## 10. Généralisation temporelle et robustesse — expérience à construire

**Question : les relations apprises restent-elles utiles sur une période ultérieure ?** La séparation par adresse de la référence ne répond pas à cette question.

### 10.1 Fenêtres chronologiques

Définir un historique d'entraînement, une période suivante de validation, puis une période ultérieure d'évaluation. Répéter cette organisation sur plusieurs fenêtres, en veillant à disposer de positifs suffisants. Comparer un historique croissant à un historique récent de longueur fixe, car les campagnes et les pratiques évoluent.

L'article utilise des fenêtres historiques pour prédire des journées ultérieures et retire des adresses déjà connues dans certaines expériences. Notre protocole devra indiquer séparément les résultats sur les adresses déjà rencontrées et sur les nouvelles adresses. Aucune date de coupure ne sera choisie uniquement parce qu'elle donne un meilleur score.

### 10.2 Charge quotidienne, incertitude et dérive

Rapporter les alertes et les détections par journée, les adresses distinctes et la couverture de toutes les fenêtres, y compris celles sans alerte. Les jours sans positif ne permettent pas d'estimer un rappel quotidien mais restent pertinents pour mesurer les fausses alertes. La **dérive** désigne un changement des distributions ou des relations entre variables et labels au cours du temps.

Comparer les modèles sur les mêmes observations. Pour estimer l'incertitude, rééchantillonner les adresses ou des blocs temporels selon le protocole, plutôt que supposer que toutes les lignes sont indépendantes. Si un score doit être interprété comme une probabilité de risque, une étude de **calibration**, c'est-à-dire de concordance entre scores annoncés et fréquences observées, sera nécessaire sur une population adaptée.

Le CSV constitue une population sélectionnée ; sa prévalence et ses volumes de `white` ne représentent pas automatiquement toute la blockchain. Les performances et charges mesurées ne doivent donc pas être extrapolées sans nouvelles données représentatives.


## 11. Familles inconnues et méthodes alternatives — extension de l'étude

**Question : peut-on signaler des observations d'une famille dont aucun exemple n'a servi à apprendre ?** Cette question ne consiste pas à prédire le nom d'une famille encore inconnue.

### 11.1 Évaluation par famille tenue à l'écart

Pour chaque famille suffisamment représentée, retirer ses observations de l'entraînement et du réglage ; entraîner sur les autres familles et les `white`, puis mesurer sa détection. Pour revendiquer une découverte à l'apparition, respecter aussi la chronologie et la disponibilité réelle des annotations. Le rappel par famille et les effectifs doivent accompagner toute moyenne.

### 11.2 Détection d'anomalies et comparaison à l'article

Comparer les meilleurs modèles supervisés à Isolation Forest, LOF et One-Class SVM. Ces méthodes recherchent des observations atypiques ou extérieures à une population de référence ; une anomalie n'est pas automatiquement un rançongiciel. Leur entraînement sur des `white` demande une discussion sur la contamination possible de cette population.

DBSCAN et Mapper/TDA pourront compléter cette extension si les ressources le permettent. Dans le tableau VII de l'article, RF et XGBoost échouent dans la configuration de nouvelle famille considérée. Ce résultat justifie une expérience spécifique ; il ne démontre pas l'échec de ces algorithmes sur toute classification supervisée. Les abstentions, les fenêtres couvertes et les effectifs doivent être comparés avec les précisions et rappels.

### 11.3 Autres références possibles

Un arbre seul peut fournir une référence lisible. Naive Bayes fournit une référence rapide, avec des hypothèses de distribution et d'indépendance à discuter. Un SVM pourra être étudié sur un échantillon adapté avec prétraitement appris sur train. Les méthodes de projection et de clustering servent à explorer des structures ; la présence d'un groupe visuel ne démontre pas une classe malveillante. Aucune multiplication des algorithmes ne remplace un protocole commun.


## 12. Conclusion et décisions suivantes

La première forêt utilise les six caractéristiques d'activité pour prédire les annotations disponibles sur des adresses non vues. Elle retrouve **657 des 7 919 positifs** du test exploratoire, avec **1 734 fausses alertes**. Son exactitude élevée masque un faible rappel. L'AP supérieure à la prévalence de référence suggère un signal de classement à examiner ; elle ne suffit pas à valider un usage opérationnel.

**Décisions proposées.** Conserver cette expérience comme référence ; mesurer le compromis du seuil sur une validation distincte ; comparer les pondérations et la régularisation de Random Forest ; évaluer les deux ratios ; comparer ensuite XGBoost dans les mêmes conditions. La confirmation temporelle et l'expérience sur familles inconnues restent deux étapes distinctes, nécessaires avant toute conclusion plus large.

**Informations à demander.** La provenance et la date d'attribution des labels, la définition exacte des fenêtres, la couverture des observations `white`, la disponibilité des transactions brutes et la capacité réelle d'investigation conditionnent la portée de l'étude. Des exemples supplémentaires vérifiés permettraient d'examiner les alertes en désaccord avec `white`.

Le résultat attendu est un choix argumenté de modèle et de seuil pour un objectif défini, avec une charge d'investigation, une stabilité et des limites mesurées. À ce stade, la détection de toutes les activités de rançongiciels sur Bitcoin n'est pas démontrée.


## Références méthodologiques

- Akcora, Li, Gel et Kantarcioglu, *BitcoinHeist: Topological Data Analysis for Ransomware Detection on the Bitcoin Blockchain*, fichier [article.pdf](article.pdf), notamment les sections IV à VI et les tableaux V et VII. [Version des auteurs](https://arxiv.org/abs/1906.07852).
- Documentation de [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), pour l'agrégation, les réglages et les conversions numériques.
- Documentation du [réglage du seuil](https://scikit-learn.org/stable/modules/classification_threshold.html), pour séparer apprentissage des scores et choix d'une décision.
- Documentation du [réglage de XGBoost](https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html), pour la complexité, la régularisation et le déséquilibre.


## Annexe A — Préparation de la restitution et repères pédagogiques

*Cette annexe sert à l'équipe et peut être retirée d'une version destinée au client.*

### Couverture de la grille d'évaluation

| Critère | Emplacement et éléments à présenter |
|---|---|
| Méthodologie /4 | Partie 1 : problème et hypothèses ; parties 5, 8 et 10 : choix des modèles et protocoles |
| Pédagogie /4 | Définitions au fil du texte, figures en partie 3, matrice et courbe en partie 5 ; analyse après chaque figure |
| Réflexion /5 | Exploration (3), feature engineering (4), apprentissage (5), tuning expliqué et expérience de seuil (6–7), conclusion et demandes d'information (12) |
| Notions /4 | AI/ML/DL et supervisé (1), corrélation (3), feature et normalisation (4), hyperparamètre et overfitting (5–7), non supervisé (11) |
| Clarté vidéo /3 | Un objectif par séquence ; figure lisible ; constat puis conséquence ; limite explicite |


### Apport des chapitres 1 à 14

| Chapitre | Contribution retenue ou limite d'application |
|---|---|
| 1 — Non supervisé | Projection, clustering et densité pour explorer ; les groupes ne sont pas des preuves de familles |
| 2 — Texte | Pas de texte dans le CSV : pas de TF-IDF sans source supplémentaire ; les principes de prétraitement restent utiles |
| 3 — SVM, biais/variance et noyaux | Comparaison possible, normalisation et coût à considérer ; cadre de généralisation |
| 4 — Multi-label | Une seule étiquette fournie par observation : pas de formulation multi-label nécessaire |
| 5 — Naive Bayes | Référence rapide ; indépendance conditionnelle et distributions à discuter |
| 6 — Processus gaussiens | Régression et classification possibles ; coût limitant sur plusieurs millions de lignes |
| 7 — Optimisation bayésienne | Allocation des essais d'hyperparamètres lorsque les entraînements sont coûteux |
| 8 — Arbres | Coupures par entropie, interprétation et sensibilité aux données |
| 9 — Boosting | Principe des modèles séquentiels et de la réduction d'une perte |
| 10 — Gradient boosting | XGBoost, learning rate, régularisation et sous-échantillonnage |
| 11 — Bagging | Bootstrap et réduction de variance par agrégation |
| 12 — Random Forest | Diversification des arbres, importance et dispersion entre arbres |
| 13 — Anomalies | Isolation Forest, LOF et One-Class SVM pour l'extension à des comportements atypiques |
| 14 — Explicabilité | LIME/SHAP pour expliquer les prédictions sans conclure à une causalité |

### Fil conducteur pour la vidéo

Réserver environ 10 % du temps au problème, 20 % à l'exploration, 15 % aux caractéristiques, 20 % à l'apprentissage, 20 % à l'amélioration/comparaison et 15 % à la conclusion. Montrer les paramètres de la forêt et les figures dans le notebook ; ouvrir les fonctions auxiliaires uniquement si leur logique est nécessaire à l'explication. Pour chaque résultat : question posée, observation, conséquence, limite.
